> **Internship:** Celebal Technologies Excellence Internship Program  
> **Domain:** Data Engineering  
> **Week:** 6  
> **Topic:** PySpark Architecture and Components
> **Author:** Gaurav Kumar

# Week 6 Assignment  
## PySpark Architecture and Components

# Introduction

Apache Spark has become one of the most widely adopted frameworks for big data processing due to its speed, scalability, and ability to process massive datasets efficiently. Unlike traditional MapReduce, Spark performs most computations in memory, significantly improving the performance of iterative and analytical workloads. Its distributed architecture enables organizations to process terabytes of data while maintaining fault tolerance and high availability.

This notebook explores the fundamental concepts of Spark Architecture, including the Driver, Cluster Manager, Executors, Lazy Evaluation, Lineage Graph (DAG), and the execution model that enables efficient distributed computing. It also demonstrates practical DataFrame operations such as reading data from CSV files, filtering records, selecting and transforming columns, handling different data types, and understanding the performance benefits of the Parquet file format.

The hands-on exercises in this assignment simulate common data engineering tasks performed in real-world ETL pipelines, helping build practical skills required for developing scalable and optimized big data solutions using Apache Spark.

## Project Setup


In [ ]:
%pip install pyspark findspark pandas matplotlib jupyter

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week-06-Spark-Assignment") \
    .getOrCreate()

df = spark.read.csv(
    "data/pyspark-assignment-dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+--------------+--------+------------+--------+-------------------+
|user_id|transaction_date|product_id|   category|  price|base_price|  amount|quantity|   status|priority|region|     city|         email|username|subscription|store_id|      raw_timestamp|
+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+--------------+--------+------------+--------+-------------------+
|   1163|      2025-01-29|      P328|Electronics|4405.65|    3733.6|  8811.3|       2|Completed|     Low| North|   Mumbai|user1@mail.com|  user_1|    Standard|     S07|2025-01-09 00:05:00|
|   1055|      2025-03-01|      P303|      Books|3598.87|   3049.89|17994.35|       5|Cancelled|     Low|  West|  Kolkata|user2@mail.com|  user_2|       Basic|     S10|2025-03-13 00:48:00|
|   1040|      2025-06-28|      P320|  Groceries|2085.3

## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Answer

Apache Spark follows a distributed architecture where different components work together to process large datasets efficiently. The three main components are the **Driver**, **Cluster Manager**, and **Executors**.

### 1. Driver
The Driver is the main control process of a Spark application. It creates the SparkSession, converts user code into execution plans (DAG), schedules tasks, and coordinates the execution of the application.

**Responsibilities:**
- Creates the SparkSession.
- Builds the execution plan (DAG).
- Divides jobs into stages and tasks.
- Schedules tasks to Executors.
- Collects and returns the final results.

### 2. Cluster Manager
The Cluster Manager is responsible for managing the cluster resources. It allocates CPU cores and memory to the Spark application and launches Executors on available worker nodes.

**Responsibilities:**
- Allocates resources for Spark applications.
- Launches Executor processes.
- Monitors available cluster resources.
- Supports cluster managers such as Standalone, YARN, Mesos, and Kubernetes.

### 3. Executor
Executors are worker processes that execute the tasks assigned by the Driver. They perform computations, store intermediate data in memory, and send the results back to the Driver.

**Responsibilities:**
- Execute tasks assigned by the Driver.
- Perform data processing operations.
- Cache intermediate data in memory.
- Return execution results to the Driver.

### Workflow

1. The Driver starts the Spark application.
2. The Driver requests resources from the Cluster Manager.
3. The Cluster Manager launches Executors on worker nodes.
4. The Driver assigns tasks to the Executors.
5. Executors process the data and return the results to the Driver.

```
                 Spark Application

                      Driver
                         │
             Requests Resources
                         │
                  Cluster Manager
                 /              \
                /                \
          Executor 1         Executor 2
          (Worker Node)      (Worker Node)
               │                  │
         Process Data       Process Data
               │                  │
                \                /
                 \              /
              Results → Driver
              
```

### Conclusion

The Driver controls the execution of the Spark application, the Cluster Manager manages cluster resources, and the Executors perform the actual data processing. Together, these components enable Spark to execute large-scale distributed data processing efficiently.

---

## Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Answer

**Lazy Evaluation** is one of the key optimization techniques used by Apache Spark. Instead of executing each transformation immediately, Spark records all transformations and waits until an **Action** (such as `show()`, `collect()`, or `count()`) is called. It then creates an optimized execution plan before processing the data.

This approach allows Spark to combine multiple transformations into a single execution pipeline, reducing unnecessary computations, disk I/O, and data movement across the cluster.

### Working of Lazy Evaluation

1. The user applies multiple transformations (e.g., `filter()`, `select()`, `groupBy()`).
2. Spark does not execute these transformations immediately.
3. Instead, it builds a **Lineage Graph (DAG)** representing all the operations.
4. When an Action is invoked, Spark optimizes the DAG and executes only the required computations.

### Example

```python
df = spark.read.csv("sales.csv", header=True, inferSchema=True)

result = (df
          .filter(df.amount > 1000)
          .select("customer_id", "amount")
          .groupBy("customer_id")
          .sum("amount"))

result.show()
```

In the above example, the `filter()`, `select()`, and `groupBy()` operations are **transformations** and are not executed immediately. Spark waits until `show()` is called, then optimizes the entire workflow and executes it efficiently.

### Flow Diagram of Lazy Evaluation

```
Transformations

filter()
     ↓
select()
     ↓
groupBy()
     ↓
sum()

(No Execution Yet)
        │
        ▼
Action → show()

        │
        ▼
Spark Optimizes DAG

        │
        ▼
Actual Execution
```

### Advantages of Lazy Evaluation

- Improves execution performance by optimizing the complete query.
- Reduces unnecessary disk I/O and data movement.
- Minimizes memory usage by executing only required operations.
- Combines multiple transformations into a single optimized execution plan.
- Enables better fault tolerance through the Lineage Graph (DAG).

### Conclusion

Lazy Evaluation improves Spark's performance by delaying execution until an action is called. This enables Spark to optimize the complete sequence of transformations, resulting in faster execution, efficient resource utilization, and better performance when processing large datasets.

## Q3. Write a Spark command to read a CSV file located at `"data/source.csv"`, ensuring the first row is treated as a header and `inferSchema` is enabled.

### Answer

In Apache Spark, the `read.csv()` method is used to load CSV files into a DataFrame. By setting `header=True`, Spark treats the first row as column names, and `inferSchema=True` automatically detects the data type of each column.

### Code

In [5]:
df = spark.read.csv(
    "data/pyspark-assignment-dataset.csv",
    header=True,
    inferSchema=True
)
df.printSchema()
df.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)

+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+--------------+--------+------------+--------+-------------------+
|user_id|transaction_date|product_id|   category|  price|base_price|  amount|quantity|   status|priority|region|     c

### Explanation

- `spark.read.csv()` reads the CSV file and creates a Spark DataFrame.
- `header=True` specifies that the first row contains column names.
- `inferSchema=True` automatically identifies the appropriate data types (such as Integer, Double, or String) instead of treating all columns as strings.

### Conclusion

Using `header=True` and `inferSchema=True` simplifies data loading by automatically assigning column names and detecting data types, reducing manual preprocessing and making the dataset ready for analysis.

## Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

### Answer

CSV and Parquet are two commonly used file formats in Apache Spark, but they differ significantly in how they store data and their performance characteristics.

- **CSV (Comma-Separated Values)** is a **row-based** file format where data is stored row by row in plain text. It is easy to read and widely supported but occupies more storage space and requires more time to process large datasets.

- **Parquet** is a **columnar** file format that stores data column by column. It is highly optimized for analytical workloads because Spark reads only the required columns instead of scanning the entire dataset.

### Difference Between CSV and Parquet

| Feature | CSV | Parquet |
|---------|-----|----------|
| Storage Format | Row-based | Columnar |
| File Size | Larger | Smaller (Compressed) |
| Schema Support | No | Yes |
| Compression | Limited | Built-in Compression |
| Read Performance | Slower | Faster |
| Best Use Case | Data Exchange | Big Data Analytics |

### Why Does It Matter for Performance?

Parquet improves performance because Spark reads only the required columns instead of the entire dataset. This reduces disk I/O, memory usage, and network transfer, making analytical queries significantly faster. Additionally, Parquet supports efficient compression and predicate pushdown, further improving query execution speed.

### Diagram of Row-based vs Columnar Storage

                CSV (Row-Based)

Row1 → ID | Name | Price | Category
Row2 → ID | Name | Price | Category
Row3 → ID | Name | Price | Category


             Parquet (Columnar)

ID        → 1 | 2 | 3 | 4
Name      → A | B | C | D
Price     → 100 | 200 | 150 | 180
Category  → E | F | G | H

### Conclusion

CSV is simple and suitable for data sharing, whereas Parquet is optimized for big data processing. In Apache Spark, Parquet is generally preferred because it provides better compression, faster query execution, and efficient storage, making it ideal for large-scale data engineering and analytics.

## Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the category is `'Electronics'`.

### Answer

In Apache Spark, the `filter()` method is used to retrieve rows that satisfy a specified condition, while the `select()` method is used to choose only the required columns from the DataFrame.

### Code

In [6]:
from pyspark.sql.functions import col

result = (df
          .filter(col("category") == "Electronics")
          .select("product_id", "price"))

result.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P328|4405.65|
|      P489|5063.45|
|      P646|3639.99|
|      P612|5632.97|
|      P446|1551.39|
|      P919|2445.95|
|      P971|5673.23|
|      P846|3567.38|
|      P619|4279.62|
|      P498|1313.92|
|      P777|2125.53|
|      P399|5751.38|
|      P561|1023.07|
|      P906|4092.46|
|      P709| 462.83|
|      P358|2747.81|
|      P918| 284.92|
|      P554|4514.46|
|      P416|5341.48|
|      P238|1475.61|
+----------+-------+
only showing top 20 rows


### Explanation

- `filter(col("category") == "Electronics")` selects only those records where the **category** is **Electronics**.
- `select("product_id", "price")` returns only the **product_id** and **price** columns.
- `show()` displays the filtered result in a tabular format.

### Conclusion

Using `filter()` together with `select()` allows Spark to retrieve only the required rows and columns, reducing unnecessary data processing and improving query efficiency.

## Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

### Answer

In Apache Spark, the `withColumnRenamed()` method is used to rename an existing column, while the `cast()` function is used to convert a column from one data type to another. In this example, the column **old_name** is renamed to **new_name**, and the **price** column is converted from **StringType** to **DoubleType** for numerical computations.

### Code

In [8]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

df_updated = (df
              .withColumnRenamed("old_name", "new_name")
              .withColumn("price", col("price").cast(DoubleType())))

df.printSchema()

df_updated.printSchema()

df_updated.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- subscription: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: in

### Explanation

- `withColumnRenamed("old_name", "new_name")` renames the existing column.
- `withColumn()` creates or updates a column in the DataFrame.
- `cast(DoubleType())` converts the `price` column from **String** to **Double**.
- `show(5)` displays the first five rows of the updated DataFrame.

### Conclusion

Renaming columns improves readability, while casting data types ensures that numerical data can be used correctly for calculations and analytical operations in Spark.

## Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

### Answer

Apache Spark provides **fault tolerance** through the **Lineage Graph**, also known as the **Directed Acyclic Graph (DAG)**. Instead of storing multiple copies of intermediate data, Spark keeps track of all the transformations applied to a dataset. This sequence of transformations forms the Lineage Graph.

If a worker node (Executor) fails and some data is lost, Spark does not restart the entire application. Instead, it uses the Lineage Graph to identify the lost partition and recomputes only that missing data from the original source using the recorded transformations.

### Working of the Lineage Graph

1. Spark reads data from the source.
2. Every transformation (such as `filter()`, `select()`, or `groupBy()`) is recorded in the Lineage Graph.
3. When an Action is executed, Spark processes the transformations and generates the required output.
4. If an Executor fails, Spark refers to the Lineage Graph.
5. Only the lost partition is recomputed, while the remaining processed data is reused.

### Advantages of Using the Lineage Graph

- Provides automatic fault tolerance without manual recovery.
- Recomputes only the lost partitions instead of the entire dataset.
- Reduces recovery time and improves efficiency.
- Eliminates the need to replicate intermediate data, saving storage space.
- Ensures reliable distributed data processing.

### Illustration

```text
Input Data
     │
     ▼
 filter()
     │
     ▼
 select()
     │
     ▼
 groupBy()
     │
     ▼
 sum()
     │
     ▼
 Final Output

(Lineage Graph records every transformation)

If an Executor fails,
Spark recomputes only the missing partition
using the recorded transformations.
```

### Conclusion

Spark achieves fault tolerance by maintaining a **Lineage Graph (DAG)** that records every transformation applied to a dataset. When a worker node fails, Spark uses this graph to recompute only the lost data instead of reprocessing the entire dataset, making distributed processing faster, more reliable, and highly efficient.

## Q8. Write a query to filter a DataFrame `df_orders` for rows where the `status` is `'Completed'` **AND** the `amount` is greater than `1000`.

### Answer

In Apache Spark, multiple filtering conditions can be combined using logical operators such as **AND (`&`)** and **OR (`|`)**. In this query, the DataFrame is filtered to retrieve only those orders where the **status** is **Completed** and the **amount** is greater than **1000**.

### Code


In [11]:
from pyspark.sql.functions import col

result = df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)
result.show(10)

result.count()


+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+---------------+--------+------------+--------+-------------------+
|user_id|transaction_date|product_id|   category|  price|base_price|  amount|quantity|   status|priority|region|     city|          email|username|subscription|store_id|      raw_timestamp|
+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+---------------+--------+------------+--------+-------------------+
|   1163|      2025-01-29|      P328|Electronics|4405.65|    3733.6|  8811.3|       2|Completed|     Low| North|   Mumbai| user1@mail.com|  user_1|    Standard|     S07|2025-01-09 00:05:00|
|   1056|      2025-06-25|      P334|  Furniture|4991.36|   4229.97| 4991.36|       1|Completed|  Medium|  West|   Jaipur| user7@mail.com|  user_7|     Premium|     S04|2025-05-26 22:20:00|
|   1092|      2025-02-26|      P873|   Clothing|3

326

## Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

### Answer

**Predicate Pushdown** is an optimization technique used by Apache Spark when reading **Parquet** files. Instead of loading the entire dataset into memory and then applying filter conditions, Spark pushes the filter (predicate) down to the Parquet storage layer. As a result, only the rows that satisfy the specified condition are read into memory.

This optimization significantly reduces disk I/O, memory usage, and query execution time, especially when working with large datasets.

### Example

```python
df = spark.read.parquet("sales.parquet")

result = df.filter(df.region == "North")
```

In the above example, Spark reads only the data blocks containing records where the **region** is **North**, instead of scanning the entire Parquet file.

### Advantages of Predicate Pushdown

- Reduces the amount of data read from disk.
- Loads only the required records into memory.
- Improves query execution speed.
- Reduces CPU and memory consumption.
- Enhances the performance of analytical queries on large datasets.

### How It Improves Performance

Without Predicate Pushdown:

```
Parquet File
      │
      ▼
Read Entire File
      │
      ▼
Load into Memory
      │
      ▼
Apply Filter
```

With Predicate Pushdown:

```
Parquet File
      │
      ▼
Apply Filter While Reading
      │
      ▼
Read Only Required Data
      │
      ▼
Load into Memory
```

### Conclusion

Predicate Pushdown is a powerful optimization feature of Parquet that allows Spark to read only the required data based on filter conditions. This minimizes memory usage, reduces disk I/O, and significantly improves the performance of big data processing.

## Q10. Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by **1.18 (18% tax)**.

### Answer

In Apache Spark, the `withColumn()` method is used to create a new column or modify an existing one. In this example, a new column named **final_price** is created by multiplying the **base_price** by **1.18**, representing the addition of **18% tax**.

### Code

In [12]:
from pyspark.sql.functions import col

df_updated = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

df_updated.show(5)

+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+--------------+--------+------------+--------+-------------------+------------------+
|user_id|transaction_date|product_id|   category|  price|base_price|  amount|quantity|   status|priority|region|     city|         email|username|subscription|store_id|      raw_timestamp|       final_price|
+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+--------------+--------+------------+--------+-------------------+------------------+
|   1163|      2025-01-29|      P328|Electronics|4405.65|    3733.6|  8811.3|       2|Completed|     Low| North|   Mumbai|user1@mail.com|  user_1|    Standard|     S07|2025-01-09 00:05:00| 4405.647999999999|
|   1055|      2025-03-01|      P303|      Books|3598.87|   3049.89|17994.35|       5|Cancelled|     Low|  West|  Kolkata|user2@mail.com|  user_2|       Basic|     S10|

### Explanation

- `withColumn()` creates a new column named **final_price**.
- `col("base_price")` retrieves the values from the **base_price** column.
- Multiplying by **1.18** adds **18% tax** to the original price.
- `show(5)` displays the first five rows of the updated DataFrame.

### Conclusion

The `withColumn()` method allows new columns to be created using existing columns and expressions. It is commonly used in ETL pipelines for deriving new features, performing calculations, and preparing data for analysis.

## Q11. What is the difference between Transformations and Actions? Provide two examples of each.

### Answer

In Apache Spark, operations are broadly classified into **Transformations** and **Actions**.

- **Transformations** create a new DataFrame or RDD from an existing one without executing the computation immediately. They are evaluated lazily and are executed only when an Action is called.

- **Actions** trigger the execution of all pending transformations and either return a result to the Driver or write the processed data to external storage.

### Difference Between Transformations and Actions

| Feature | Transformations | Actions |
|---------|-----------------|----------|
| Execution | Lazy (Not executed immediately) | Triggers execution |
| Output | Creates a new DataFrame/RDD | Returns a result or writes data |
| Purpose | Defines data processing steps | Produces the final output |
| Optimization | Included in the DAG optimization | Executes the optimized DAG |

### Examples of Transformations

1. **filter()** – Selects rows that satisfy a specified condition.

```python
df_filtered = df.filter(df.amount > 1000)
```

2. **select()** – Selects specific columns from a DataFrame.

```python
df_selected = df.select("product_id", "price")
```

### Examples of Actions

1. **show()** – Displays the contents of a DataFrame.

```python
df.show()
```

2. **count()** – Returns the total number of rows in a DataFrame.

```python
df.count()
```

### Explanation

When a transformation such as `filter()` or `select()` is executed, Spark does not process the data immediately. Instead, it records these operations in the **Lineage Graph (DAG)**.

Only when an Action such as `show()` or `count()` is called does Spark optimize the execution plan and perform the actual computation.

### Workflow

```
Read Data
     │
     ▼
filter()      ← Transformation
     │
     ▼
select()      ← Transformation
     │
     ▼
groupBy()     ← Transformation
     │
     ▼
show()        ← Action
     │
     ▼
Spark Executes the DAG
```

### Conclusion

Transformations define how data should be processed, while Actions trigger the actual execution of those transformations. Spark's Lazy Evaluation allows it to optimize all transformations before executing them, resulting in efficient and high-performance distributed data processing.

## Q12. Write the Spark command to load a Parquet file from `"path/to/input"`, filter out any rows where `user_id` is null, and save the result as a CSV at `"path/to/output"`.

### Answer

In Apache Spark, the `read.parquet()` method is used to load Parquet files into a DataFrame. The `filter()` method removes rows with null values in the `user_id` column, and the `write.csv()` method saves the cleaned DataFrame as a CSV file.

### Code

```python
from pyspark.sql.functions import col

# Read the Parquet file
df = spark.read.parquet("path/to/input")

# Filter rows where user_id is not null
df_clean = df.filter(col("user_id").isNotNull())

# Save the cleaned DataFrame as a CSV file
df_clean.write.mode("overwrite").option("header", True).csv("path/to/output")
```

### Explanation

- `spark.read.parquet()` loads the Parquet file into a Spark DataFrame.
- `filter(col("user_id").isNotNull())` removes records where the `user_id` is null.
- `write.mode("overwrite")` replaces the existing output folder if it already exists.
- `option("header", True)` writes the column names as the first row in the CSV file.
- `csv("path/to/output")` saves the cleaned data in CSV format.

### Workflow

```
Read Parquet File
        │
        ▼
Filter Null user_id
        │
        ▼
Create Clean DataFrame
        │
        ▼
Save as CSV
```

### Conclusion

This process demonstrates a common ETL workflow in Apache Spark: reading data from a Parquet file, cleaning the dataset by removing invalid records, and exporting the processed data as a CSV file for further analysis or sharing.

## Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

### Answer

Apache Spark applications can run in **Client Mode** or **Cluster Mode**, depending on where the **Driver Program** is executed.

- In **Client Mode**, the Driver runs on the machine from which the Spark application is submitted (client machine). The Executors run on the worker nodes in the cluster. This mode is commonly used during development, testing, and debugging because the Driver logs are directly available on the local machine.

- In **Cluster Mode**, the Driver runs inside the cluster on one of the worker nodes, while the client machine only submits the application. This mode is generally used in production environments because the application continues running even if the client disconnects.

### Difference Between Client Mode and Cluster Mode

| Feature | Client Mode | Cluster Mode |
|---------|-------------|--------------|
| Driver Location | Client Machine | Cluster Node |
| Executors | Worker Nodes | Worker Nodes |
| Suitable For | Development & Testing | Production |
| Client Dependency | Client must remain connected | Client can disconnect after submission |
| Fault Tolerance | Lower | Higher |

### Working

**Client Mode**

```
Client Machine
     │
 Driver Program
     │
Cluster Manager
     │
Executors on Worker Nodes
```

**Cluster Mode**

```
Client Machine
     │
Submits Application
     │
Cluster Manager
     │
Driver Program (Inside Cluster)
     │
Executors on Worker Nodes
```

### Advantages

**Client Mode**
- Easy to debug applications.
- Driver logs are available on the local machine.
- Suitable for development and testing.

**Cluster Mode**
- Better fault tolerance.
- Application continues running even if the client disconnects.
- Ideal for long-running production jobs.

### Conclusion

The primary difference between Client Mode and Cluster Mode is the location of the Driver Program. Client Mode is preferred for development and debugging, whereas Cluster Mode is recommended for production workloads due to its better reliability, scalability, and fault tolerance.

## Q14. Write a query to filter a dataset for rows where the `region` is `'North'` **OR** the `priority` is `'High'`.

### Answer

In Apache Spark, the `filter()` method is used to retrieve rows that satisfy one or more conditions. Multiple conditions can be combined using logical operators such as **AND (`&`)** and **OR (`|`)**. In this query, the DataFrame is filtered to return records where the **region** is **North** or the **priority** is **High**.

### Code

In [14]:
from pyspark.sql.functions import col

result = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

result.show()

+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+---------------+--------+------------+--------+-------------------+
|user_id|transaction_date|product_id|   category|  price|base_price|  amount|quantity|   status|priority|region|     city|          email|username|subscription|store_id|      raw_timestamp|
+-------+----------------+----------+-----------+-------+----------+--------+--------+---------+--------+------+---------+---------------+--------+------------+--------+-------------------+
|   1163|      2025-01-29|      P328|Electronics|4405.65|    3733.6|  8811.3|       2|Completed|     Low| North|   Mumbai| user1@mail.com|  user_1|    Standard|     S07|2025-01-09 00:05:00|
|   1040|      2025-06-28|      P320|  Groceries|2085.33|   1767.23| 4170.66|       2|  Pending|    High| North|    Patna| user3@mail.com|  user_3|     Premium|     S06|2025-03-30 19:16:00|
|   1179|      2025-06-24|      P646|Electronics|3

### Explanation

- `col("region") == "North"` selects records belonging to the **North** region.
- `col("priority") == "High"` selects records with **High** priority.
- The `|` operator combines both conditions using the logical **OR** operator.
- `show()` displays the filtered records.

### Conclusion

The `filter()` function, combined with the **OR (`|`)** operator, allows Spark to retrieve records that satisfy **at least one** of the specified conditions. This type of filtering is widely used in data engineering to extract relevant subsets of data for reporting and analysis.

## Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

### Answer

When working with large datasets in Apache Spark, it is safer to use **`.show(5)`** because it displays only the first **five rows** of the DataFrame without transferring the entire dataset to the Driver program.

In contrast, **`.collect()`** retrieves **all rows** from the distributed cluster and loads them into the Driver's memory. For multi-terabyte datasets, this can consume excessive memory, significantly slow down the application, or even cause an **OutOfMemoryError**.

### Difference Between `.show(5)` and `.collect()`

| Feature | `.show(5)` | `.collect()` |
|---------|------------|--------------|
| Data Retrieved | First 5 rows | Entire dataset |
| Memory Usage | Very Low | Very High |
| Suitable for Large Datasets | Yes | No |
| Risk of Driver Crash | Very Low | High |
| Primary Use | Data Preview | Retrieve complete dataset |

### Example

```python
# Display only the first 5 rows
df.show(5)

# Retrieve the entire dataset
data = df.collect()
```

### Explanation

- `show(5)` is mainly used to **preview** the data during exploration and debugging.
- `collect()` transfers **every row** from all Executors to the Driver, which is practical only for **small datasets**.
- On very large datasets, using `collect()` may exhaust the Driver's memory and impact application performance.

### Best Practice

- Use **`.show()`** to inspect or verify data.
- Use **`.collect()`** only when the dataset is small enough to fit comfortably into the Driver's memory.
- For large-scale data processing, perform computations using Spark transformations instead of bringing the entire dataset to the Driver.

### Conclusion

For multi-terabyte datasets, **`.show(5)`** is a safer and more efficient choice because it retrieves only a small sample of the data. In contrast, **`.collect()`** loads the entire dataset into the Driver's memory, making it unsuitable for large-scale data processing due to the risk of memory exhaustion and reduced performance.

# Key Learnings

Throughout this assignment, the following key concepts of Apache Spark were learned and implemented:

- Understood the architecture of Apache Spark, including the roles of the Driver, Cluster Manager, and Executors.
- Learned how Spark's **Lazy Evaluation** optimizes execution by delaying computation until an Action is invoked.
- Explored the **Lineage Graph (DAG)** and its role in providing fault tolerance and efficient task execution.
- Understood the differences between **Client Mode** and **Cluster Mode** in Spark applications.
- Compared **CSV** and **Parquet** file formats, and learned why Parquet is preferred for big data analytics.
- Studied **Predicate Pushdown** and how it improves query performance by reducing the amount of data read into memory.
- Performed various DataFrame operations, including filtering, selecting columns, renaming columns, casting data types, and creating derived columns.
- Learned the difference between **Transformations** and **Actions** and how they contribute to Spark's optimized execution model.
- Practiced reading from and writing to different file formats using Apache Spark.
- Strengthened practical understanding of Spark programming and distributed data processing concepts used in modern data engineering.

# Conclusion

This assignment provided practical exposure to Apache Spark's architecture, execution model, and DataFrame API. It covered both theoretical concepts and hands-on implementations, including Spark Architecture, Lazy Evaluation, Lineage Graph (DAG), Transformations and Actions, DataFrame operations, and file handling using CSV and Parquet formats.

The exercises demonstrated how Spark optimizes distributed data processing through techniques such as in-memory computation, predicate pushdown, and execution plan optimization. These concepts are fundamental for building scalable, fault-tolerant, and high-performance data engineering applications.

Overall, this assignment strengthened both conceptual understanding and practical skills required to develop efficient ETL pipelines and process large-scale datasets using Apache Spark in real-world environments.